# 01 Data Joining — Women's

This notebook joins all women's raw data sources into clean, unified datasets for downstream EDA, feature engineering, and modeling.

**Key differences from men's pipeline:**
- Compact results start in 1998 (vs. 1985 for men's)
- Detailed box score results start in 2010 (vs. 2003 for men's)
- No Massey Ordinals available — must derive team quality entirely from game results and box scores
- No coaches data available
- ~1.5% of games in 2010–2012 may be missing detailed results

**Inputs** (from `00_data_collection/`):
- `WRegularSeasonCompactResults.csv` — game scores, 1998–2026
- `WRegularSeasonDetailedResults.csv` — box scores, 2010–2026
- `WNCAATourneyCompactResults.csv` — tournament scores, 1998–2025
- `WNCAATourneyDetailedResults.csv` — tournament box scores, 2010–2025
- `WNCAATourneySeeds.csv` — tournament seeds
- `WTeams.csv`, `WTeamConferences.csv`, `Conferences.csv`, `WSeasons.csv`

**Outputs** (to S3 `01_data_joining/womens/`):
1. `regular_season_games.parquet` — all regular season games with detailed stats where available
2. `tourney_games.parquet` — all tournament games with detailed stats where available
3. `team_season_stats.parquet` — per-team per-season aggregated statistics
4. `tourney_seeds.parquet` — cleaned tournament seeds with numeric seed extracted
5. `team_metadata.parquet` — team info and conferences per season

In [ ]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "01_data_joining"
INPUT_PREFIX = f"s3://{BUCKET}/00_data_collection/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

# For local development, fall back to local data directory
LOCAL_DATA = "../00_data_collection/"
LOCAL_OUTPUT = "output/"

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

def read_csv(filename):
    """Read CSV from S3 if available, otherwise fall back to local."""
    try:
        return pd.read_csv(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_csv(f"{LOCAL_DATA}{filename}")

os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Raw Data

In [ ]:
# Game results
reg_compact = read_csv("WRegularSeasonCompactResults.csv")
reg_detailed = read_csv("WRegularSeasonDetailedResults.csv")
tourney_compact = read_csv("WNCAATourneyCompactResults.csv")
tourney_detailed = read_csv("WNCAATourneyDetailedResults.csv")

# Tournament metadata
seeds = read_csv("WNCAATourneySeeds.csv")
seasons = read_csv("WSeasons.csv")

# Team metadata (no coaches file for women's)
teams = read_csv("WTeams.csv")
team_conferences = read_csv("WTeamConferences.csv")
conferences = read_csv("Conferences.csv")

print(f"Regular season compact: {reg_compact.shape} (seasons {reg_compact.Season.min()}-{reg_compact.Season.max()})")
print(f"Regular season detailed: {reg_detailed.shape} (seasons {reg_detailed.Season.min()}-{reg_detailed.Season.max()})")
print(f"Tourney compact: {tourney_compact.shape}")
print(f"Tourney detailed: {tourney_detailed.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Teams: {teams.shape}")

## 2. Join Regular Season Games

Merge compact (1998–2026) and detailed (2010–2026) results. Pre-2010 seasons will have NaN for box score columns. Note that ~1.5% of 2010–2012 games may also be missing detailed results.

In [ ]:
compact_cols = reg_compact.columns.tolist()
detail_only_cols = [c for c in reg_detailed.columns if c not in compact_cols]

reg_games = reg_compact.merge(
    reg_detailed[compact_cols + detail_only_cols],
    on=compact_cols,
    how='left'
)

print(f"Regular season games: {reg_games.shape}")
print(f"Seasons with detailed stats: {reg_games.dropna(subset=['WFGM']).Season.nunique()}")
print(f"Seasons without detailed stats: {reg_games[reg_games['WFGM'].isna()].Season.nunique()}")

# Check for missing detailed results in 2010-2012 range
early_detail = reg_games[(reg_games.Season >= 2010) & (reg_games.Season <= 2012)]
missing_pct = early_detail['WFGM'].isna().mean() * 100
print(f"\nMissing detailed stats in 2010-2012: {missing_pct:.1f}%")

## 3. Join Tournament Games

In [ ]:
tourney_games = tourney_compact.merge(
    tourney_detailed[[c for c in tourney_detailed.columns]],
    on=compact_cols,
    how='left'
)

print(f"Tournament games: {tourney_games.shape}")
print(f"Tournament seasons: {tourney_games.Season.min()} - {tourney_games.Season.max()}")

## 4. Build Team-Season Aggregated Statistics

Convert winner/loser format to team-centric rows, then aggregate per team per season. Since there are no Massey Ordinals for women's, these derived stats are especially important as the primary signal for team quality.

In [ ]:
def build_team_game_rows(games_df):
    """
    Convert winner/loser format into team-centric rows.
    Each game produces two rows: one for each team.
    """
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    has_detail = 'WFGM' in games_df.columns and games_df['WFGM'].notna().any()
    
    # Winner rows
    w = games_df[['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']].copy()
    w.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    w['Win'] = 1
    w['Loc'] = w['WLoc'].map({'H': 'H', 'A': 'A', 'N': 'N'})
    
    # Loser rows
    l = games_df[['Season', 'DayNum', 'LTeamID', 'LScore', 'WTeamID', 'WScore', 'WLoc', 'NumOT']].copy()
    l.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    l['Win'] = 0
    l['Loc'] = l['WLoc'].map({'H': 'A', 'A': 'H', 'N': 'N'})
    
    if has_detail:
        for col in box_cols:
            w[col] = games_df[f'W{col}'].values
            w[f'Opp{col}'] = games_df[f'L{col}'].values
            l[col] = games_df[f'L{col}'].values
            l[f'Opp{col}'] = games_df[f'W{col}'].values
    
    w = w.drop(columns=['WLoc'])
    l = l.drop(columns=['WLoc'])
    
    return pd.concat([w, l], ignore_index=True)

team_games = build_team_game_rows(reg_games)
print(f"Team-game rows: {team_games.shape}")
team_games.head()

In [ ]:
def aggregate_team_season(team_games_df):
    """
    Compute per-team per-season aggregate stats from team-centric game rows.
    """
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    opp_box_cols = [f'Opp{c}' for c in box_cols]
    
    agg_dict = {
        'Win': ['sum', 'count'],
        'Score': 'mean',
        'OppScore': 'mean',
    }
    
    for col in box_cols + opp_box_cols:
        if col in team_games_df.columns:
            agg_dict[col] = 'mean'
    
    stats = team_games_df.groupby(['Season', 'TeamID']).agg(agg_dict)
    
    # Flatten MultiIndex columns
    stats.columns = ['_'.join(col).strip('_') for col in stats.columns]
    stats = stats.rename(columns={
        'Win_sum': 'Wins',
        'Win_count': 'Games',
        'Score_mean': 'AvgScore',
        'OppScore_mean': 'AvgOppScore',
    })
    
    for col in box_cols + opp_box_cols:
        if f'{col}_mean' in stats.columns:
            stats = stats.rename(columns={f'{col}_mean': f'Avg{col}'})
    
    stats['Losses'] = stats['Games'] - stats['Wins']
    stats['WinPct'] = stats['Wins'] / stats['Games']
    stats['AvgPointDiff'] = stats['AvgScore'] - stats['AvgOppScore']
    
    # Derived shooting percentages and efficiency (only for seasons with detailed stats)
    if 'AvgFGM' in stats.columns:
        stats['FGPct'] = stats['AvgFGM'] / stats['AvgFGA']
        stats['FG3Pct'] = stats['AvgFGM3'] / stats['AvgFGA3']
        stats['FTPct'] = stats['AvgFTM'] / stats['AvgFTA']
        stats['OppFGPct'] = stats['AvgOppFGM'] / stats['AvgOppFGA']
        stats['OppFG3Pct'] = stats['AvgOppFGM3'] / stats['AvgOppFGA3']
        
        # Possessions estimate: FGA - OR + TO + 0.475 * FTA
        stats['AvgPoss'] = (stats['AvgFGA'] - stats['AvgOR'] + stats['AvgTO'] + 0.475 * stats['AvgFTA'])
        stats['AvgOppPoss'] = (stats['AvgOppFGA'] - stats['AvgOppOR'] + stats['AvgOppTO'] + 0.475 * stats['AvgOppFTA'])
        
        # Offensive and Defensive Efficiency (points per possession)
        stats['OffEff'] = stats['AvgScore'] / stats['AvgPoss']
        stats['DefEff'] = stats['AvgOppScore'] / stats['AvgOppPoss']
        stats['NetEff'] = stats['OffEff'] - stats['DefEff']
    
    stats = stats.reset_index()
    return stats

team_season_stats = aggregate_team_season(team_games)
print(f"Team-season stats: {team_season_stats.shape}")
print(f"Columns: {team_season_stats.columns.tolist()}")

# Show how many team-seasons have efficiency stats vs. only basic stats
has_eff = team_season_stats['OffEff'].notna().sum() if 'OffEff' in team_season_stats.columns else 0
print(f"\nTeam-seasons with efficiency stats: {has_eff} / {len(team_season_stats)}")
team_season_stats.head()

## 5. Clean Tournament Seeds

Extract numeric seed from the seed string (e.g., `W01` → 1, `X16a` → 16).

In [ ]:
tourney_seeds = seeds.copy()
tourney_seeds['Region'] = tourney_seeds['Seed'].str[0]
tourney_seeds['SeedNum'] = tourney_seeds['Seed'].str[1:3].astype(int)
tourney_seeds['PlayIn'] = tourney_seeds['Seed'].str[3:]  # 'a', 'b', or ''

print(f"Tournament seeds: {tourney_seeds.shape}")
print(f"Seed range: {tourney_seeds.SeedNum.min()} - {tourney_seeds.SeedNum.max()}")
print(f"Seasons: {tourney_seeds.Season.min()} - {tourney_seeds.Season.max()}")
tourney_seeds.head()

## 6. Build Team Metadata

Join team names and conference affiliations per season. No coaches data is available for women's.

In [ ]:
# Join conferences with their descriptions
team_conf = team_conferences.merge(conferences, on='ConfAbbrev', how='left')
team_conf = team_conf.rename(columns={'Description': 'Conference'})

# Identify power conferences
power_conf_abbrevs = ['acc', 'big_ten', 'big_twelve', 'sec', 'pac_twelve', 'big_east']
team_conf['IsPowerConf'] = team_conf['ConfAbbrev'].isin(power_conf_abbrevs).astype(int)

# Join team names
team_meta = team_conf.merge(
    teams[['TeamID', 'TeamName']],
    on='TeamID',
    how='left'
)

print(f"Team metadata: {team_meta.shape}")
team_meta.head()

## 7. Save Outputs

Write all joined datasets to S3 (parquet) and local output directory. Note: no Massey Ordinals output for women's — team quality must be derived from game results and box scores in later pipeline stages.

In [ ]:
outputs = {
    'regular_season_games': reg_games,
    'tourney_games': tourney_games,
    'team_season_stats': team_season_stats,
    'tourney_seeds': tourney_seeds,
    'team_metadata': team_meta,
}

for name, df in outputs.items():
    # Save to S3
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")
    
    # Save locally
    local_path = f"{LOCAL_OUTPUT}{name}.parquet"
    df.to_parquet(local_path, index=False)
    print(f"Saved locally: {local_path} ({df.shape})")

## 8. Output Summary

In [ ]:
print("=" * 60)
print("DATA JOINING SUMMARY — WOMEN'S")
print("=" * 60)
for name, df in outputs.items():
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()[:10]}{'...' if len(df.columns) > 10 else ''}")
    if 'Season' in df.columns:
        print(f"  Seasons: {df.Season.min()} - {df.Season.max()}")

print("\n" + "=" * 60)
print("NOTE: No Massey Ordinals available for women's.")
print("Team quality must be derived from game results and box scores.")
print("Detailed box scores available from 2010 onwards only.")
print("=" * 60)